<a href="https://colab.research.google.com/github/baclayonjonrel/privacy-policy/blob/main/notebooks/piper_multilingual_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) training notebook.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)

---

- Notebook made by [rmcpantoja](http://github.com/rmcpantoja)
- Collaborator: [Xx_Nessu_xX](http://github.com/Xx_Nessu_xX)

---

# Notes:

- <font color="orange">**Things in orange mean that they are important.**

# Credits:

* [Feanix-Fyre fork](https://github.com/Feanix-Fyre/piper) with some improvements.
* [Tacotron2 NVIDIA training notebook](https://github.com/justinjohn0306/FakeYou-Tacotron2-Notebook) - Dataset duration snippet.
* [🐸TTS](https://github.com/coqui-ai/TTS) - Resampler and XTTS formater demo.

# <font color="ffc800">🔧 ***First steps.*** 🔧

In [1]:
#@markdown ## <font color="ffc800"> **Google Colab Anti-Disconnect.** 🔌
#@markdown ---
#@markdown #### Avoid automatic disconnection. Still, it will disconnect after <font color="orange">**6 to 12 hours**</font>.

import IPython
js_code = '''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''
display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

In [2]:
#@markdown ## <font color="ffc800"> **Check GPU type.** 👁️
#@markdown ---
#@markdown #### A higher capable GPU can lead to faster training speeds. By default, you will have a <font color="orange">**Tesla T4**</font>.
!nvidia-smi

Wed May 27 07:09:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
#@markdown # <font color="ffc800"> **Mount Google Drive.** 📂
#@markdown ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---
#@markdown ####In this cell the synthesizer and its necessary dependencies to execute the training will be installed. (this may take a while)

import os

# ALWAYS shift back to a safe baseline folder before performing operations
%cd /content

# Clean up previous directory to avoid clone errors safely
if os.path.exists('/content/piper'):
    !rm -rf /content/piper

# Clone and navigate
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python
!wget -q "https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/bin/resample.py"

print("Installing core dependencies (this will take a minute)...")

# 1. Force exact library versions required for Piper training stability
!pip install --no-warn-conflicts -q \
    "cython>=0.29.0" \
    "piper-phonemize-fix" \
    "numpy==1.26.4" \
    "numba==0.60.0" \
    "librosa==0.10.1" \
    "torch==2.2.2" \
    "pytorch-lightning==2.2.2" \
    "onnxruntime>=1.15.0"

# 2. CRITICAL PYTHON 3.12 FIX: Patches the internal 'ImpImporter' metadata loader crash
!pip install --no-warn-conflicts -q setuptools==69.5.1

# 3. Install ancillary tools safely without breaking numpy/torch environment mappings
!pip install --no-warn-conflicts -q --no-deps gdown transformers

print("Building monotonic alignment modules...")
!bash build_monotonic_align.sh

# Useful vars:
use_whisper = True
print("\n Done! All system patches applied successfully.")

/content
/content/piper/src/python
Installing core dependencies (this will take a minute)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.9/801.9 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

# <font color="ffc800"> 🤖 ***Training.*** 🤖

In [5]:
#@markdown # <font color="ffc800"> **1. Extract dataset.** 📥
#@markdown ---
#@markdown ####Important: the audios must be in <font color="orange">**wav format, (16000 or 22050hz, 16-bits, mono), and, for convenience, numbered.**

import os
import wave
import zipfile
import datetime
import shutil
from glob import glob

def get_dataset_duration(wav_path):
    totalduration = 0
    wav_files = [
        x for x in os.listdir(wav_path)
        if os.path.isfile(os.path.join(wav_path, x)) and x.endswith(".wav")
    ]
    for file_name in wav_files:
        full_path = os.path.join(wav_path, file_name)
        try:
            # FIXED HERE: Removed the backslashes from around "rb"
            with wave.open(full_path, "rb") as wave_file:
                frames = wave_file.getnframes()
                rate = wave_file.getframerate()
                duration = frames / float(rate)
                totalduration += duration
        except Exception as e:
            print(f"Skipping unreadable file: {file_name}")
            continue
    wav_count = len(wav_files)
    duration_str = str(datetime.timedelta(seconds=round(totalduration, 0)))
    return wav_count, duration_str

# Base resets safely
%cd /content
dataset_dir = "/content/dataset"
wavs_dir = "/content/dataset/wavs"

if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
os.makedirs(wavs_dir)
print("Dataset folder reset complete.")

#@markdown ### Audio dataset path to unzip:
zip_path = "/content/drive/MyDrive/training_data/kimyat_wavs.zip" #@param {type:"string"}
zip_path = zip_path.strip()

if zip_path and os.path.exists(zip_path):
    if zipfile.is_zipfile(zip_path):
        print("Unzipping audio content...")
        !unzip -q -j "{zip_path}" -d /content/dataset/wavs
    else:
        print("Copying audio contents of this folder...")
        fp = zip_path + "/."
        !cp -a "$fp" "/content/dataset/wavs"
else:
    raise Exception("The path provided to the wavs is not correct. Please set a valid path.")

if os.path.exists("/content/dataset/wavs/wavs"):
    for file in os.listdir("/content/dataset/wavs/wavs"):
        !mv /content/dataset/wavs/wavs/"$file" /content/dataset/wavs/"$file"
    !rm -rf /content/dataset/wavs/wavs

for file in glob("/content/dataset/wavs/*.txt"): os.remove(file)
for file in glob("/content/dataset/wavs/*.csv"): os.remove(file)
!find /content/dataset/wavs/ -name "._*" -delete
!find /content/dataset/wavs/ -name ".DS_Store" -delete

%cd /content/dataset/wavs
audio_count, dataset_dur = get_dataset_duration("/content/dataset/wavs")
print(f"Opened dataset with {audio_count} wavs with duration {dataset_dur}.")
%cd /content

/content
Dataset folder reset complete.
Unzipping audio content...
/content/dataset/wavs
Opened dataset with 1366 wavs with duration 1:17:58.
/content


In [6]:
#@markdown # <font color="ffc800"> **2. Upload the transcript file.** 📝
#@markdown ---
%cd /content/dataset
from google.colab import files

if os.path.exists("/content/dataset/metadata.csv"):
    !rm /content/dataset/metadata.csv

if os.path.exists("/content/dataset/wavs/_transcription.txt"):
    !mv "/content/dataset/wavs/_transcription.txt" metadata.csv
else:
    uploaded = files.upload()
    if uploaded:
        listfn, length = list(uploaded.items())[0]
        if listfn != "metadata.csv":
            !mv "$listfn" metadata.csv

use_whisper = False
%cd /content

/content/dataset


Saving metadata.csv to metadata.csv
/content


In [7]:
#@markdown # <font color="ffc800"> **3. Preprocess dataset.** 🔄
#@markdown ---
import os

#@markdown ### First of all, select the language of your dataset.
language = "English (U.S.)" #@param ["ألعَرَبِي", "Català", "čeština", "Dansk", "Deutsch", "Ελληνικά", "English (British)", "English (U.S.)", "Español (Castellano)", "Español (Latinoamericano)", "Suomi", "Français", "Magyar", "Icelandic", "Italiano", "ქართული", "қазақша", "Lëtzebuergesch", "नेपाली", "Nederlands", "Norsk", "Polski", "Português (Brasil)", "Português (Portugal)", "Română", "Русский", "Српски", "Svenska", "Kiswahili", "Türkçe", "украї́нська", \"Tiếng Việt\", "简体中文"]

languages = {
    "ألعَرَبِي": "ar", "Català": "ca", "čeština": "cs", "Dansk": "da", "Deutsch": "de",
    "Ελληνικά": "el", "English (British)": "en", "English (U.S.)": "en-us", "Español (Castellano)": "es",
    "Español (Latinoamericano)": "es-419", "Suomi.": "fi", "Français": "fr", "Magyar": "hu",
    "Icelandic": "is", "Italiano": "it", "ქართული": "ka", "қазақша": "kk", "Lëtzebuergesch": "lb",
    "नेपाली": "ne", "Nederlands": "nl", "Norsk": "nb", "Polski": "pl", "Português (Brasil)": "pt-br",
    "Português (Portugal)": "pt-pt", "Română": "ro", "Русский": "ru", "Српски": "sr",
    "Svenska": "sv", "Kiswahili": "sw", "Türkçe": "tr", "украї́нська": "uk", "Tiếng Việt": "vi", "简体中文": "zh"
}

final_language = "es-419"

#@markdown ### Choose a name for your model:
model_name = "Pedro_Ceb" #@param {type:"string"}

#@markdown ### Choose the working folder:
output_path = "/content/drive/MyDrive/colab/piper" #@param {type:"string"}
output_dir = output_path+"/"+model_name

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

#@markdown ### Choose dataset format:
dataset_format = "ljspeech" #@param ["ljspeech", "mycroft"]

#@markdown ### Is this a single speaker dataset?
single_speaker = True #@param {type:"boolean"}
force_sp = " --single-speaker" if single_speaker else ""

#@markdown ### Select the sample rate of the dataset:
sample_rate = "22050" #@param ["16000", "22050"]

if not os.path.exists("/content/audio_cache"):
    os.makedirs("/content/audio_cache")

%cd /content/piper/src/python

resample = False #@param {type:"boolean"}
if resample:
    !python resample.py --input_dir "/content/dataset/wavs" --output_dir "/content/dataset/wavs_resampled" --output_sr {sample_rate} --file_ext "wav"
    !mv /content/dataset/wavs_resampled/* /content/dataset/wavs

# Run Preprocess via the explicitly localized PYTHONPATH context
!PYTHONPATH=/content/piper/src/python python -m piper_train.preprocess \
  --language {final_language} \
  --input-dir /content/dataset \
  --cache-dir "/content/audio_cache" \
  --output-dir "{output_dir}" \
  --dataset-name "{model_name}" \
  --dataset-format {dataset_format} \
  --sample-rate {sample_rate} \
  {force_sp}

print("Preprocessing done!")

/content/piper/src/python
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 1366 utterance(s) with 2 worker(s)
Preprocessing done!


In [ ]:
#@markdown # <font color="ffc800"> **4. Settings.** 🧰
#@markdown ---
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import output
import os
import re
import glob
#@markdown ### <font color="orange">**Select the action to train this dataset: (READ CAREFULLY)**

#@markdown * The option to <font color="orange">continue a training</font> is self-explanatory. If you've previously trained a model with free colab, your time is up and you're considering training it some more, this is ideal for you. You just have to set the same settings that you set when you first trained this model.
#@markdown * The option to <font color="orange">convert a single-speaker model to a multi-speaker model</font> is self-explanatory, and for this it is important that you have processed a dataset that contains text and audio from all possible speakers that you want to train in your model.
#@markdown * The <font color="orange">finetune</font> option is used to train a dataset using a pretrained model, that is, train on that data. This option is ideal if you want to train a very small dataset (more than five minutes recommended).
#@markdown * The <font color="orange">train from scratch</font> option builds features such as dictionary and speech form from scratch, and this may take longer to converge. For this, hours of audio (8 at least) are recommended, which have a large collection of phonemes.

action = "Continue training" #@param ["Continue training", "convert single-speaker to multi-speaker model", "finetune", "train from scratch"]
#@markdown ---
if action == "Continue training":
    checkpoints = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
    if len(checkpoints):
        last_checkpoint = sorted(checkpoints, key=lambda x: int(re.findall(r'version_(\d+)', x)[0]))[-1]
        ft_command = f'--resume_from_checkpoint "{last_checkpoint}" '
        print(f"Continuing {model_name}'s training at: {last_checkpoint}")
    else:
        raise Exception("Training cannot be continued as there is no checkpoint to continue at.")
elif action == "finetune":
    if os.path.exists(f"{output_dir}/lightning_logs/version_0/checkpoints/last.ckpt"):
        raise Exception("Oh no! You have already trained this model before, you cannot choose this option since your progress will be lost, and then your previous time will not count. Please select the option to continue a training.")
    else:
        ft_command = '--resume_from_checkpoint "/content/pretrained.ckpt" '
elif action == "convert single-speaker to multi-speaker model":
    if not single_speaker:
        ft_command = '--resume_from_single_speaker_checkpoint "/content/pretrained.ckpt" '
    else:
        raise Exception("This dataset is not a multi-speaker dataset!")
else:
    ft_command = ""
if action== "convert single-speaker to multi-speaker model" or action == "finetune":
    try:
        with open('/content/piper/notebooks/pretrained_models.json') as f:
            pretrained_models = json.load(f)
        if final_language in pretrained_models:
            models = pretrained_models[final_language]
            model_options = [(model_name, model_name) for model_name, model_url in models.items()]
            model_dropdown = widgets.Dropdown(description = "Choose pretrained model", options=model_options)
            download_button = widgets.Button(description="Download")
            def download_model(btn):
                model_name = model_dropdown.value
                model_url = pretrained_models[final_language][model_name]
                print("\033[93mDownloading pretrained model...")
                if model_url.startswith("1"):
                    !gdown -q "{model_url}" -O "/content/pretrained.ckpt"
                elif model_url.startswith("https://drive.google.com/file/d/"):
                    !gdown -q "{model_url}" -O "/content/pretrained.ckpt" --fuzzy
                else:
                    !wget -q "{model_url}" -O "/content/pretrained.ckpt"
                model_dropdown.close()
                download_button.close()
                output.clear()
                if os.path.exists("/content/pretrained.ckpt"):
                    print("\033[93mModel downloaded!")
                else:
                    raise Exception("Couldn't download the pretrained model!")
            download_button.on_click(download_model)
            display(model_dropdown, download_button)
        else:
            raise Exception(f"There are no pretrained models available for the language {final_language}")
    except FileNotFoundError:
        raise Exception("The pretrained_models.json file was not found.")
else:
    print("\033[93mWarning: this model will be trained from scratch. You need at least 8 hours of data for everything to work decent. Good luck!")
#@markdown ### Choose batch size based on this dataset:
batch_size = 16 #@param {type:"integer"}
#@markdown ---

#@markdown ### Choose the quality for this model:

#@markdown * x-low - 16Khz audio, 5-7M params
#@markdown * medium - 22.05Khz audio, 15-20 params
#@markdown * high - 22.05Khz audio, 28-32M params
quality = "medium" #@param ["high", "x-low", "medium"]
#@markdown ---
#@markdown ### For how many epochs to save training checkpoints?
#@markdown The larger your dataset, you should set this saving interval to a smaller value, as epochs can progress longer time.
checkpoint_epochs = 5 #@param {type:"integer"}
#@markdown ---
#@markdown ### Interval to save best k models:
#@markdown Set to 0 if you want to disable saving multiple models. If this is the case, check the checkbox below. If set to 1, models will be saved with the file name epoch=xx-step=xx.ckpt, so you will need to empty Drive's trash every so often.
num_ckpt = 1 #@param {type:"integer"}
#@markdown ---
#@markdown ### Save latest model:
#@markdown This checkbox must be checked if you want to save a single model (last.ckpt). Saving a single model is applied only if num_ckpt is equal to 0. If so, the interval parameter of epochs to save is ignored, since the last model per epoch is saved; also, you won't have to worry about storage. Being equal to 1, last.ckpt will be saved, but another model (model_vVersion.ckpt, the latter takes into account the epoch range you set), so you would have to empty the trash often.

#@markdown **It's not recommended to use this option in extremely small datasets, since by saving the last model each epoch, this process will be very fast and the trainer will not be able to save the complete model, which would result in a corrupt last.ckpt.**
save_last = True # @param {type:"boolean"}
#@markdown ---
#@markdown ### Step interval to generate model samples:
log_every_n_steps = 1000 #@param {type:"integer"}
#@markdown ---
#@markdown ### Training epochs:
max_epochs = 11000 #@param {type:"integer"}
#@markdown ---

In [ ]:
#@markdown # <font color="ffc800"> **5. Run the TensorBoard extension.** 📈
#@markdown ---
#@markdown The TensorBoard is used to visualize the results of the model while it's being trained such as audio and losses.

%load_ext tensorboard
%tensorboard --logdir {output_dir}

In [ ]:
#@markdown # <font color="ffc800"> **6. Train.** 🏋️‍♂️
#@markdown ---
#@markdown ### Run this cell to train your final model!

import os

#@markdown ---
#@markdown ### <font color="orange">**Disable validation?**
validation = True #@param {type:"boolean"}

#@markdown ### <font color="orange">**Save Last Checkpoint?**
save_last = True #@param {type:"boolean"}

if validation:
    validation_split = 0.01
    num_test_examples = 1
else:
    validation_split = 0
    num_test_examples = 0

if not save_last:
    save_last_command = ""
else:
    save_last_command = "--save_last True "

os.makedirs(output_dir, exist_ok=True)

# Run training inside the proper path context
get_ipython().system(f'''
PYTHONPATH=/content/piper/src/python \
NUMPY_EXPERIMENTAL_ARRAY_FUNCTION=0 \
python -m piper_train \
--dataset-dir "{output_dir}" \
--accelerator 'gpu' \
--devices 1 \
--batch-size {batch_size} \
--validation-split {validation_split} \
--num-test-examples {num_test_examples} \
--quality {quality} \
--checkpoint-epochs {checkpoint_epochs} \
--num_ckpt {num_ckpt} \
{save_last_command}\
--log_every_n_steps {log_every_n_steps} \
--max_epochs {max_epochs} \
{ft_command}\
--precision 32
''')

In [ ]:
# 1. Uninstall the conflicting versions
!pip uninstall -y torch torchvision torchmetrics pytorch-lightning

# 2. Install matching, stable versions (PyTorch 2.1 or 2.2 series works best with older Piper setups)
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1

# 3. Install compatible versions of lightning and metrics
!pip install pytorch-lightning==2.2.1 torchmetrics==1.3.1

# 4. (Optional) Re-run your piper setup/requirements if needed
# !pip install -e .

In [ ]:
# Create the structural dependencies and compile directly via standard Cython distribution layout
%cd /content/piper/src/python/piper_train/vits/monotonic_align

import os
import shutil

# 1. Clean the environment of broken C templates and existing targets
for file in ["core.c", "core.pyx.c"]:
    if os.path.exists(file):
        os.remove(file)

# 2. CRITICAL STEP: Create the directory layout inside itself before running setup.py
# This prevents the "No such file or directory" compiler destination drop failure
os.makedirs("piper_train/vits/monotonic_align", exist_ok=True)
os.makedirs("monotonic_align", exist_ok=True)

print("Compiling Python 3.12 binary extensions...")
# 3. Compile manually with modern language constraints explicitly declared
!cythonize -i -3 core.pyx
!python setup.py build_ext --inplace

# 4. Safely locate the binary extension and link it to both possible patterns
compiled_files = [f for f in os.listdir(".") if f.startswith("core") and f.endswith(".so")]

if compiled_files:
    for file in compiled_files:
        # Link to modern directory lookups
        shutil.copy(file, os.path.join("monotonic_align", file))
        # Link to hardcoded absolute lookup pathways
        shutil.copy(file, os.path.join("piper_train/vits/monotonic_align", file))
    print("\n Success! Monotonic Alignment modules compiled and mirrored for Python 3.12.")
else:
    # Secondary check inside nested structure if fallback occurred
    print("\nExtracting from sub-build trees...")
    !find . -name "core*.so" -exec cp {} ./monotonic_align/ \;
    !find . -name "core*.so" -exec cp {} ./piper_train/vits/monotonic_align/ \;
    print("Linked and secured.")

%cd /content

#  <font color="orange">**Have you finished training and want to test the model?**

* If you want to run this model in any software that Piper integrates or the same Piper app, export your model using the [model exporter notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_model_exporter.ipynb)!
* Wait! I want to test this right now before exporting it to the supported format for Piper. Test your generated last.ckpt with [this notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ckpt).ipynb)!